# AutoAug-like Search vs Adaptive Rule Policy

Ноутбук сравнивает два подхода для темы диплома:

- **Adaptive rule policy**: анализирует датасет, строит интерпретируемую policy и запускает baseline/manual/adaptive/ablation.
- **AutoAug-like budget search**: генерирует несколько случайных candidate policies, обучает их в одинаковом бюджете и выбирает лучшую по `AP_small`.

Это не полноценный AutoAugment/RL-поиск. Это честный low-compute сравнитель, который показывает цену поиска по сравнению с быстрым data-driven выбором политики.

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd()
if (REPO / 'src').exists() and str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

PROJECT_CONFIG = 'configs/project_config.yaml'
TRAIN_PROFILE = 'fast'  # fast for pilot, final for dissertation-grade runs
NUM_AUTOAUG_CANDIDATES = 6
SEED = 42

## 1. Adaptive Pipeline

Запускаем основной pipeline проекта. Для быстрого пилота можно оставить `run_training=False`, чтобы получить только stats/policy. Для полного сравнения поставьте `run_training=True` и `run_eval=True`.

In [ ]:
from src.pipeline_mvp import run_mvp_pipeline

adaptive_outputs = run_mvp_pipeline(
    project_config_path=PROJECT_CONFIG,
    run_training=False,
    run_eval=False,
    train_profile=TRAIN_PROFILE,
)
adaptive_outputs

## 2. Generate AutoAug-like Candidates

Кандидаты сохраняются как YAML-файлы. Каждый YAML можно запускать как manual policy с тем же бюджетом обучения.

In [ ]:
from src.experiments.autoaug_search import generate_autoaug_candidates, save_autoaug_candidates

candidates = generate_autoaug_candidates(num_candidates=NUM_AUTOAUG_CANDIDATES, seed=SEED)
candidate_paths = save_autoaug_candidates(candidates)
candidate_paths

In [ ]:
import pandas as pd

pd.DataFrame([
    {'candidate_id': c.candidate_id, **c.ultralytics_args}
    for c in candidates
])

## 3. Optional: Train AutoAug Candidates

Эта ячейка запускает обучение каждого кандидата. В Colab/GPU включайте её осознанно: стоимость равна `NUM_AUTOAUG_CANDIDATES × TRAIN_PROFILE`.

In [ ]:
RUN_AUTOAUG_TRAINING = False

if RUN_AUTOAUG_TRAINING:
    from src.training.train_runner import TrainRunConfig, run_train_mode
    from src.pipeline_mvp import _resolve_training_profile, _write_runtime_data_yaml, _dataset_class_names
    from src.utils.io import load_yaml
    
    cfg = load_yaml(PROJECT_CONFIG)
    dataset_root = Path(cfg['dataset']['root'])
    training_cfg = cfg['training']
    profile_cfg = _resolve_training_profile(training_cfg, TRAIN_PROFILE)
    class_names = _dataset_class_names(cfg, dataset_root)
    runtime_yaml = _write_runtime_data_yaml(dataset_root, class_names, 'artifacts/runtime_autoaug.yaml')
    
    train_cfg = TrainRunConfig(
        data_yaml=runtime_yaml.as_posix(),
        model=training_cfg['model'],
        epochs=profile_cfg['epochs'],
        imgsz=profile_cfg['imgsz'],
        batch=profile_cfg['batch'],
        device=training_cfg.get('device'),
        workers=profile_cfg['workers'],
        fraction=profile_cfg['fraction'],
        project_dir='runs/autoaug_search',
        seed=SEED,
        deterministic=cfg['project'].get('deterministic', True),
        rect=False,
        multi_scale=False,
    )
    
    autoaug_run_dirs = {}
    for candidate in candidates:
        autoaug_run_dirs[candidate.candidate_id] = run_train_mode(
            mode=candidate.candidate_id,
            config=train_cfg,
            mode_args=candidate.ultralytics_args,
            custom_augmentations=None,
        ).as_posix()
    autoaug_run_dirs
else:
    print('Training skipped. Set RUN_AUTOAUG_TRAINING=True on a GPU runtime.')

## 4. Report Template

После обучения и оценки заполняется таблица. Основная метрика диплома: `AP_small`; обязательный контекст: количество training trials и время.

In [ ]:
comparison = pd.DataFrame([
    {
        'approach': 'adaptive_rule_policy',
        'policy_selection_cost': 'dataset analysis + rule engine',
        'training_trials': 'baseline/manual/adaptive/2 ablations',
        'primary_metric': 'AP_small via COCOeval',
        'artifacts': adaptive_outputs.get('mvp_report', adaptive_outputs.get('decision_report')),
    },
    {
        'approach': 'autoaug_like_random_search',
        'policy_selection_cost': f'{NUM_AUTOAUG_CANDIDATES} candidate trainings',
        'training_trials': NUM_AUTOAUG_CANDIDATES,
        'primary_metric': 'best candidate AP_small via COCOeval',
        'artifacts': candidate_paths['manifest'],
    },
])
comparison